<a href="https://colab.research.google.com/github/Icey-Python/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Professional RAG Application: Constitution of Kenya
This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) pipeline using Google Gemini, ChromaDB, and Sentence Transformers.

In [25]:
pip install python-gemini-api chromadb sentence_transformers

In [26]:
### Install a PDF parsing library
!pip install pymupdf

In [27]:
from google.colab import userdata

# Fetch the API key from Colab Secrets
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("API Key successfully loaded from secrets.")
except Exception:
    print("GEMINI_API_KEY not found in secrets. Please add it to the secrets sidebar.")

API Key successfully loaded from secrets.


## 1. Setup and Configuration
In this section, we install dependencies, initialize our embedding model, and connect to the vector database.

In [28]:
import os
import fitz  # PyMuPDF
import chromadb
from google import genai
from sentence_transformers import SentenceTransformer
from google.colab import drive

# 1. Initialize Gemini Client
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Load Embedding Model
# 'all-MiniLM-L6-v2' is efficient and great for local RAG tasks
embeddings = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Setup Vector Database (ChromaDB)
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="kenya_constitution",
    metadata={"description": "Indexed chunks of the Kenyan Constitution 2010"}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
client=genai.Client(api_key=GEMINI_API_KEY)

In [30]:
embeddings= SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
chroma_client=chromadb.Client()
collection =  chroma_client.get_or_create_collection(name="documents", metadata={"descriiption" : "Document store for RAG agents"})

### Loading Documents from Google Drive
First, we need to mount your Google Drive to access your files.

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Document Ingestion
We mount your Google Drive, extract text from the PDF, and split it into overlapping chunks to ensure no context is lost during the embedding process.

In [35]:
# Configuration for Chunking
CHUNK_SIZE = 600
OVERLAP = 150
STEP = CHUNK_SIZE - OVERLAP

# Path to your PDF in Google Drive
pdf_path = '/content/drive/My Drive/The_Constitution_of_Kenya_2010.pdf'

def ingest_document(path):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}. Check your Drive path.")
        return

    print(f"📖 Opening {path}...")
    doc = fitz.open(path)
    full_text = "".join([page.get_text() for page in doc])

    # Generate overlapping chunks
    chunks = [full_text[i : i + CHUNK_SIZE] for i in range(0, len(full_text), STEP)]

    # Embed chunks
    print(f"🔢 Encoding {len(chunks)} chunks using Sentence Transformers...")
    vector_embeddings = embeddings.encode(chunks).tolist()

    # Upsert to ChromaDB
    collection.add(
        embeddings=vector_embeddings,
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )
    print(f"✅ Successfully indexed {len(chunks)} chunks into the vector store.")

# Execute Ingestion
drive.mount('/content/drive')
ingest_document(pdf_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📖 Opening /content/drive/My Drive/The_Constitution_of_Kenya_2010.pdf...
🔢 Encoding 730 chunks using Sentence Transformers...
✅ Successfully indexed 730 chunks into the vector store.


## 3. The RAG Query Engine
This function retrieves the most relevant context and generates a grounded response using Gemini.

In [45]:
def ask_rag(query, n_results=3):
    # 1. Embed the query
    query_embedding = embeddings.encode([query]).tolist()

    # 2. Retrieve the top N most relevant chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    # Combine the retrieved chunks into one context block
    context_segments = results['documents'][0]
    full_context = "\n---\n".join(context_segments)

    # 3. Generate response with Gemini
    prompt = f"""Answer the user query based ONLY on the provided context.
    If the answer is not in the context, say you do not know.

    Context:
    {full_context}

    Query: {query}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text, context_segments

# @title Ask the Constitution
# @markdown Type your question below and run this cell to get an answer from the PDF.
user_query = "Explain the sovereignity of the people" # @param {type:"string"}

answer, retrieved_chunks = ask_rag(user_query)

print(f"query: {user_query}\n")
print(f"ans: \n{answer}\n")

query: Explain the sovereignity of the people

ans: 
The people may exercise their sovereign power either directly or through their democratically elected representatives.

This sovereign power is exercised at:
*   the national level; and
*   the county level.

Sovereign power under this Constitution is delegated to the following State organs, which shall perform their functions in accordance with this Constitution:
*   Parliament and the legislative assemblies in the county governments;
*   the national executive and the executive structures in the county governments; and
*   the Judiciary and independent tribunals.

